[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/fundamentos/04-redes-neuronales.ipynb)

# Redes Neuronales — De la neurona al perceptrón multicapa

Las redes neuronales son la base de toda la IA moderna.
Construimos todo desde cero: neuronas, funciones de activación, MLP y backpropagation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('NumPy', np.__version__)

## 1. La neurona artificial

In [ ]:
def neurona(entradas: np.ndarray, pesos: np.ndarray, sesgo: float) -> float:
    return float(np.dot(entradas, pesos) + sesgo)

def relu(x: float) -> float:
    return max(0.0, x)

# Clasificar temperatura como 'caliente'
entradas = np.array([37.5])
pesos    = np.array([1.0])
sesgo    = -36.5

suma      = neurona(entradas, pesos, sesgo)
activacion = relu(suma)

print(f'Temperatura : {entradas[0]}°C')
print(f'Suma ponderada: {suma}')
print(f'Activación ReLU: {activacion}')
print(f'Decisión: {"caliente" if activacion > 0 else "normal"}')

## 2. Funciones de activación

In [ ]:
x = np.linspace(-5, 5, 200)

activaciones = {
    'ReLU':    np.maximum(0, x),
    'Sigmoid': 1 / (1 + np.exp(-x)),
    'Tanh':    np.tanh(x),
    'GELU':    x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3))) / 2,
}

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, (nombre, valores) in zip(axes.flat, activaciones.items()):
    ax.plot(x, valores, linewidth=2, color='#2196F3')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(nombre, fontweight='bold')
    ax.set_xlim(-5, 5)
    ax.grid(alpha=0.3)

plt.suptitle('Funciones de activación más comunes', fontsize=13)
plt.tight_layout()
plt.show()

print('Uso en la práctica:')
print('  ReLU    → CNNs y redes clásicas')
print('  GELU    → Transformers (GPT, BERT, Claude)')
print('  Sigmoid → Clasificación binaria (capa final)')
print('  Tanh    → RNNs y LSTMs')

## 3. Perceptrón multicapa (MLP) — implementación desde cero

In [ ]:
class RedNeuronal:
    """MLP con una capa oculta, inicialización Xavier."""

    def __init__(self, n_entrada: int, n_oculta: int, n_salida: int):
        self.W1 = np.random.randn(n_entrada, n_oculta) * np.sqrt(2 / n_entrada)
        self.b1 = np.zeros(n_oculta)
        self.W2 = np.random.randn(n_oculta, n_salida) * np.sqrt(2 / n_oculta)
        self.b2 = np.zeros(n_salida)

    def relu(self, x: np.ndarray) -> np.ndarray:
        return np.maximum(0, x)

    def softmax(self, x: np.ndarray) -> np.ndarray:
        e = np.exp(x - np.max(x))
        return e / e.sum()

    def adelante(self, x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        a1 = self.relu(x @ self.W1 + self.b1)
        a2 = self.softmax(a1 @ self.W2 + self.b2)
        return a1, a2

    def predecir(self, x: np.ndarray) -> int:
        _, prob = self.adelante(x)
        return int(np.argmax(prob))


np.random.seed(0)
red = RedNeuronal(n_entrada=5, n_oculta=8, n_salida=3)
clases = ['consulta_legal', 'queja', 'informacion']

textos_ejemplo = [
    (np.array([0.2, 0.8, 0.1, 0.9, 0.1]), 'consulta_legal'),
    (np.array([0.9, 0.1, 0.8, 0.1, 0.9]), 'queja'),
    (np.array([0.5, 0.3, 0.2, 0.5, 0.3]), 'informacion'),
]

print('Predicciones (pesos aleatorios, sin entrenar):')
for features, etiqueta in textos_ejemplo:
    pred = red.predecir(features)
    _, probs = red.adelante(features)
    print(f'  Real: {etiqueta:<16} Predicho: {clases[pred]:<16} Confianza: {max(probs):.1%}')

## 4. Backpropagation — cómo aprende la red

In [ ]:
def entrenar_epoch(red: RedNeuronal, X: np.ndarray, y: np.ndarray, lr: float = 0.01) -> float:
    perdida_total = 0.0
    for xi, yi in zip(X, y):
        a1, a2 = red.adelante(xi)
        y_oh = np.eye(3)[yi]
        perdida_total += -np.sum(y_oh * np.log(a2 + 1e-8))

        dL_da2 = a2 - y_oh
        dL_dW2 = np.outer(a1, dL_da2)
        dL_db2 = dL_da2
        dL_da1 = dL_da2 @ red.W2.T
        dL_da1[a1 <= 0] = 0
        dL_dW1 = np.outer(xi, dL_da1)
        dL_db1 = dL_da1

        red.W2 -= lr * dL_dW2
        red.b2 -= lr * dL_db2
        red.W1 -= lr * dL_dW1
        red.b1 -= lr * dL_db1

    return perdida_total / len(X)


np.random.seed(42)
X_train = np.random.randn(150, 5)
y_train = np.where(X_train[:, 3] > 0.5, 0, np.where(X_train[:, 2] > 0.3, 1, 2))

red = RedNeuronal(5, 16, 3)
perdidas = []

for epoch in range(50):
    p = entrenar_epoch(red, X_train, y_train, lr=0.05)
    perdidas.append(p)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}: pérdida = {p:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(perdidas, color='#4CAF50', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Pérdida (cross-entropy)')
plt.title('Curva de entrenamiento')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Comparativa de escala: MLP → LLMs

In [ ]:
modelos = {
    'MLP (ejemplo)':               {'params': 5*16 + 16 + 16*3 + 3, 'año': '1986'},
    'LeNet-5 (CNN)':               {'params': 60_000,               'año': '1998'},
    'ResNet-50':                   {'params': 25_000_000,           'año': '2015'},
    'GPT-2':                       {'params': 1_500_000_000,        'año': '2019'},
    'GPT-3':                       {'params': 175_000_000_000,      'año': '2020'},
    'Claude 3.5 Sonnet (est.)':    {'params': 200_000_000_000,      'año': '2024'},
}

def fmt_params(p: int) -> str:
    if p >= 1e9: return f'{p/1e9:.1f}B'
    if p >= 1e6: return f'{p/1e6:.1f}M'
    if p >= 1e3: return f'{p/1e3:.1f}K'
    return str(p)

print(f'{"Modelo":<35} {"Parámetros":>12} {"Año":>6}')
print('-' * 56)
for nombre, info in modelos.items():
    print(f'{nombre:<35} {fmt_params(info["params"]):>12} {info["año"]:>6}')

print('\nLos LLMs son MLPs enormes con arquitectura Transformer.')
print('La diferencia clave: mecanismo de atención (ver notebook 05).')

## 6. PyTorch — el mismo clasificador en producción

In [ ]:
try:
    import torch
    import torch.nn as nn

    class ClasificadorTexto(nn.Module):
        def __init__(self, n_entrada: int, n_oculta: int, n_clases: int):
            super().__init__()
            self.red = nn.Sequential(
                nn.Linear(n_entrada, n_oculta),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(n_oculta, n_oculta // 2),
                nn.ReLU(),
                nn.Linear(n_oculta // 2, n_clases),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.red(x)

    modelo = ClasificadorTexto(768, 256, 5)
    n_params = sum(p.numel() for p in modelo.parameters())
    print(f'Parámetros: {n_params:,}')

    with torch.no_grad():
        x = torch.randn(1, 768)
        probs = torch.softmax(modelo(x), dim=-1)
        print(f'Probabilidades: {probs.numpy().round(3)}')
        print(f'Clase predicha: {probs.argmax().item()}')

except ImportError:
    print('PyTorch no instalado. Instalar con: pip install torch')
    print('El ejemplo muestra nn.Sequential con Dropout — patrón estándar en producción.')